# Ingest Source Data

Load JSON files from the landing lakehouse and append them to the raw Delta table.

In [ ]:
control_table = "raw._sys_ingestion_config"
run_log_table = "raw._sys_run_log"
source_filter = "example_source"

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

config_df = spark.table(control_table).filter(
    (F.col("enabled") == True) & (F.col("source_name") == source_filter)
)
configs = config_df.collect()
if not configs:
    raise ValueError(f"No enabled ingestion configuration found for {source_filter}")

run_started_at = datetime.now(timezone.utc).isoformat()
for config in configs:
    source_path = config["source_path"]
    source_format = config["source_format"]
    target_table = config["target_table"]

    if not notebookutils.fs.exists(source_path):
        raise FileNotFoundError(f"Source path does not exist: {source_path}")

    source_df = spark.read.format(source_format).load(source_path)
    ingested_df = (
        source_df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_name", F.lit(config["source_name"]))
        .withColumn("_source_path", F.lit(source_path))
    )
    ingested_df.write.mode("append").format("delta").saveAsTable(target_table)

    row_count = ingested_df.count()
    run_record = spark.createDataFrame([(
        config["source_name"], target_table, run_started_at, "Succeeded", row_count
    )], ["source_name", "target_table", "run_started_at", "status", "row_count"])
    run_record.write.mode("append").format("delta").saveAsTable(run_log_table)
    print(f"Loaded {row_count} rows into {target_table}")